# Ham or Spam?

🎯 The goal of this task is to classify emails as **spam (1)** or **ham (0)**.

🧹 First, **cleaning** techniques will be applied to the text data.

👩🏻‍🔬 Then, the cleaned texts will be converted into a **numerical representation**.

✉️ Finally, a ***Multinomial Naive Bayes*** model will be applied to classify each email as spam or ham.


## (0) NLTK Library (Natural Language Toolkit)

In [2]:
!pip install nltk

In [3]:
# When importing nltk for the first time, we also need to download a few built-in libraries.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /Users/yaren/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /Users/yaren/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /Users/yaren/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /Users/yaren/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/yaren/nltk_data...


True

In [4]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) Cleaning the (Text) Dataset

The dataset consists of emails classified as ham [0] or spam [1]. You need to clean the dataset before training the prediction model.

### (1.1) Remove Punctuation

❓ Create a function to remove punctuation. Apply it to the `text` column and store the output in a new column of the dataframe called `clean_text`. ❓

In [7]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['clean_text'] = df['text'].apply(remove_punctuation)
df['clean_text'].head()

0    Subject naturally irresistible your corporate ...
1    Subject the stock trading gunslinger  fanny is...
2    Subject unbelievable new homes made easy  im w...
3    Subject 4 color printing special  request addi...
4    Subject do not have money  get software cds fr...
Name: clean_text, dtype: object

### (1.2) Lowercase

❓ Create a function to convert text to lowercase. Apply it to `clean_text`. ❓

In [11]:
def to_lowercase(text):
    return text.lower()

df['clean_text'] = df['clean_text'].apply(to_lowercase)
df['clean_text'].head()

0    subject naturally irresistible your corporate ...
1    subject the stock trading gunslinger  fanny is...
2    subject unbelievable new homes made easy  im w...
3    subject 4 color printing special  request addi...
4    subject do not have money  get software cds fr...
Name: clean_text, dtype: object

### (1.3) Remove Numbers

❓ Create a function to remove numbers from text. Apply it to `clean_text`. ❓

In [14]:
import re

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

df['clean_text'] = df['clean_text'].apply(remove_numbers)
df['clean_text'].head()

0    subject naturally irresistible your corporate ...
1    subject the stock trading gunslinger  fanny is...
2    subject unbelievable new homes made easy  im w...
3    subject  color printing special  request addit...
4    subject do not have money  get software cds fr...
Name: clean_text, dtype: object

### (1.4) Remove StopWords

❓ Create a function to remove stop words from text. Apply it to `clean_text`. ❓

In [19]:
from nltk.corpus import stopwords

def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    return ' '.join([word for word in text.split() if word not in stop_words])

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
df['clean_text'].head()

0    subject naturally irresistible corporate ident...
1    subject stock trading gunslinger fanny merrill...
2    subject unbelievable new homes made easy im wa...
3    subject color printing special request additio...
4    subject money get software cds software compat...
Name: clean_text, dtype: object

### (1.5) Lemmatize

❓ Create a function to lemmatize the text. Make sure the output is a single string, not a list of words. Apply it to `clean_text`. ❓

In [20]:
from nltk.stem import WordNetLemmatizer

def lemmatize(text):
    lemmatizer = WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

df['clean_text'] = df['clean_text'].apply(lemmatize)
df['clean_text'].head()

0    subject naturally irresistible corporate ident...
1    subject stock trading gunslinger fanny merrill...
2    subject unbelievable new home made easy im wan...
3    subject color printing special request additio...
4    subject money get software cd software compati...
Name: clean_text, dtype: object

## (2) Bag-of-Words Modeling

### (2.1) Converting Text Data to Numbers

❓ Vectorize `clean_text` into a Bag-of-Words representation using the default CountVectorizer. Save it as `X_bow`. ❓

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(df['clean_text'])

pd.DataFrame(X_bow.toarray(), columns=vectorizer.get_feature_names_out()).head()

,aa,aaa,aaaenerfax,aadedeji,aagrawal,aal,aaldous,aaliyah,aall,aanalysis,...,zwzm,zxghlajf,zyban,zyc,zygoma,zymg,zzmacmac,zzn,zzncacst,zzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### (2.2) Multinomial Naive Bayes Modeling

❓ Cross-validate the MultinomialNB model with the bag-of-words data. Score the model's accuracy. ❓

In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

model = MultinomialNB()

scores = cross_val_score(model, X_bow, df['spam'], cv=5, scoring='accuracy')

print(f'Accuracy scores: {scores}')
print(f'Mean accuracy: {scores.mean():.4f}')

Accuracy scores: [0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Mean accuracy: 0.9895
